# Day 15 · Real embeddings: let a model choose the numbers

**First-principles question:** When a model picks the numbers, what do they capture, and what do they miss?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import numpy as np
import matplotlib.pyplot as plt
from lab.llm import embed

# 10 sentences from the help articles, and 10 customer-style paraphrases (same order)
ARTICLES = [
    "A late fee of ₹100 is added if the bill is not paid by the 10th.",
    "Share the outage ticket ID; never promise an exact time for the fix.",
    "Restart the router by switching it off for 30 seconds.",
    "Cancellation needs 30 days of notice.",
    "Upgrades to a faster plan are active within 1 hour.",
    "Shifting your connection is free within our Pune service areas.",
    "Change the Wi-Fi password in the Acme app under Wi-Fi settings.",
    "Acme staff never ask for OTPs or passwords.",
    "Double payments are refunded automatically within 5 to 7 working days.",
    "A technician visit is free and happens within 2 working days.",
]
PARAPHRASES = [
    "why did you charge me 100 rupees extra",
    "when will my internet come back",
    "my router is acting weird, what do I do",
    "how long before my connection is closed if I quit",
    "I want more speed, how fast can you switch me",
    "we are moving house, can the internet come with us",
    "how do I set a new wifi password",
    "someone called and wanted my one-time code",
    "I paid my bill twice by mistake",
    "can someone come and check my line",
]

## 1 · Embed all 20 sentences

In [ ]:
# TODO: vectors = a numpy array of embed(ARTICLES + PARAPHRASES)   (shape: 20 × dimensions)
vectors = None
print(vectors.shape)
# TODO: print the length (np.linalg.norm) of the first 3 vectors. Are they 1?

## 2 · The similarity matrix

In [ ]:
# TODO: S[i, j] = cosine similarity of vectors i and j, for all pairs
#       Hint: normalise every row to length 1, then S = unit @ unit.T
S = None

labels = [f"A{i}" for i in range(10)] + [f"P{i}" for i in range(10)]
plt.figure(figsize=(7, 6)); plt.imshow(S, cmap="Blues"); plt.colorbar()
plt.xticks(range(20), labels, fontsize=7); plt.yticks(range(20), labels, fontsize=7)
plt.title("A = article sentence, P = customer paraphrase"); plt.show()

## 3 · Does each paraphrase find its article?

In [ ]:
# TODO: for each paraphrase i (row 10 + i of S), the best article = argmax of S[10 + i, :10]
best = None          # a list of 10 article numbers
top1 = None          # share of paraphrases whose best article is the right one (i)
print("best article per paraphrase:", best)
print(f"top-1 accuracy: {top1:.0%}")

## 4 · Hinglish
Write 3 paraphrases in Hinglish, and the number of the article each one belongs to.

In [ ]:
HINGLISH = [
    # TODO: ("net kab tak chalega bhai", 1),
]
h_vectors = np.array(embed([text for text, _ in HINGLISH]))
h_unit = h_vectors / np.linalg.norm(h_vectors, axis=1, keepdims=True)
a_unit = vectors[:10] / np.linalg.norm(vectors[:10], axis=1, keepdims=True)
for (text, right), row in zip(HINGLISH, h_unit @ a_unit.T):
    print(f"{text!r}: best {row.argmax()} (right answer {right}) · score {row.max():.2f}")

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert vectors.shape[0] == 20, "Step 1: 20 vectors"
assert S.shape == (20, 20) and np.allclose(S, S.T, atol=1e-6), "Step 2: S is 20 × 20 and symmetric"
assert np.allclose(np.diag(S), 1, atol=1e-3), "Step 2: every sentence is identical to itself"
assert len(best) == 10 and 0 <= top1 <= 1
assert len(HINGLISH) >= 3, "Step 4: write 3 Hinglish paraphrases"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")